In [29]:
from dotenv import load_dotenv
# from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent

load_dotenv()


True

In [30]:
PRODUCTS = {
    "wireless headphones": {"price": 79.99,  "rating": 4.6, "description": "Over-ear Bluetooth, 30-hr battery, active noise cancellation."},
    "smart watch":         {"price": 199.99, "rating": 4.3, "description": "Tracks heart rate and sleep. 5-day battery, water-resistant."},
    "mechanical keyboard": {"price": 129.00, "rating": 4.8, "description": "Tenkeyless, Cherry MX Brown switches, per-key RGB."},
    "laptop stand":        {"price": 34.99,  "rating": 4.5, "description": "Adjustable aluminium, fits 11-17 inch laptops, folds flat."},
}

@tool
def get_product(name: str) -> str:
    """Look up a product by name and return its price, rating, stock, and description."""
    p = PRODUCTS.get(name.lower())
    if not p:
        return f"Product not found. Available: {', '.join(PRODUCTS)}"
    return str(p)

In [31]:
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

agent = create_agent(
    llm,
    tools=[get_product],
    system_prompt="You are a helpful product assistant for an online tech store.",
)

In [32]:
def ask(question: str):
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    print(result["messages"][-1].content)

In [33]:
ask("what is the price of wireless headphones.")

The price of the wireless headphones is $79.99.


In [34]:
ask("tell me about mechanical keyboard")

The mechanical keyboard is a high-quality product with a price of $129. It has a rating of 4.8 out of 5 stars, indicating excellent customer satisfaction. The keyboard features Cherry MX Brown switches, which are known for their tactile bump and smooth actuation. Additionally, it has per-key RGB lighting, allowing for customizable backlighting. The tenkeyless design makes it compact and ideal for gamers or typists who prefer a more minimalist setup.


In [35]:
REVIEWS = {
    "wireless headphones": {"reviews": 1262, "rating": 4.6},
    "smart watch":         {"reviews": 340,  "rating": 3.9},
    "mechanical keyboard": {"reviews": 67,   "rating": 4.8},
    "laptop stand":        {"reviews": 781,  "rating": 4.5},
}

@tool
def get_review(name: str) -> str:
    """Look up a product review by a product name. Return the product name, number of reviews and rating"""
    r = REVIEWS.get(name.lower())
    if not r:
        return f"Review not available for this product"
    return str(r)

In [36]:
# llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

agent2 = create_agent(
    llm,
    tools=[get_product, get_review],
    system_prompt="You are a helpful product assistant for an online tech store.",
)

def ask2(question: str):
    result = agent2.invoke({"messages": [{"role": "user", "content": question}]})
    print(result["messages"][-1].content)

In [37]:
ask2("how do people like smart watch")

People generally like smartwatches, with an average rating of 4.3 out of 5 stars based on 340 reviews. They are priced at $199.99 and have features such as heart rate and sleep tracking, a 5-day battery life, and are water-resistant.


In [38]:
ask2("what is the price and reviews of this product")

The prices of the products are: 
- Wireless headphones: $79.99
- Smart watch: $199.99
- Mechanical keyboard: $129.00
- Laptop stand: $34.99

The reviews of the products are: 
- Wireless headphones: 1262 reviews, 4.6 rating
- Smart watch: 340 reviews, 3.9 rating
- Mechanical keyboard: 67 reviews, 4.8 rating
- Laptop stand: 781 reviews, 4.5 rating


In [47]:
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()

agent3 = create_agent(
    llm,
    tools=[get_product, get_review],
    system_prompt="You are a helpful product assistant.",
    checkpointer=memory
)

In [48]:
def ask3(question: str):
    result = agent3.invoke(
        {
            "messages": [("user", question)]   # ✅ tuple format (IMPORTANT)
        },
        config={
            "configurable": {
                "thread_id": "user-1"
            }
        }
    )

    print(result["messages"][-1].content)

In [49]:
ask3("how do people like smart watch")

People generally like smartwatches, with an average rating of 4.3 out of 5 stars based on 340 reviews. They are priced at $199.99 and have features such as tracking heart rate and sleep, a 5-day battery life, and are water-resistant.


In [50]:
ask3("what is the price and reviews of this product")

The prices of the products are: 
- Wireless headphones: $79.99
- Smart watch: $199.99
- Mechanical keyboard: $129.00
- Laptop stand: $34.99

The reviews of the products are: 
- Wireless headphones: 1262 reviews, 4.6 rating
- Smart watch: 340 reviews, 3.9 rating
- Mechanical keyboard: 67 reviews, 4.8 rating
- Laptop stand: 781 reviews, 4.5 rating


In [54]:
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# LLM
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# Memory
memory = InMemorySaver()

# Agent
agent4 = create_agent(
    llm,
    tools=[get_product, get_review],
    system_prompt="You are a helpful assistant.",
    checkpointer=memory
)

# def ask4(q):
#     result = agent4.invoke(
#         {"messages": [("user", q)]},   # 👈 MUST be tuple format
#         config={"configurable": {"thread_id": "test-user"}}  # 👈 MUST be SAME
#     )
#     print(result["messages"][-1].content)

def ask4(question: str):
    result = agent4.invoke(
        {
            "messages": [("user", question)]   # ✅ tuple format (IMPORTANT)
        },
        config={
            "configurable": {
                "thread_id": "user-1"
            }
        }
    )

    print(result["messages"][-1].content)



In [55]:
ask4("My name is Somnath")

Hello Somnath, it's nice to meet you. Is there something I can help you with or would you like to chat?


In [56]:
ask4("What is my name?")

Your name is Somnath.


In [ ]:
ask4("how do people like smart watch")

In [ ]:
ask4("what is the price and reviews of this product")